## Cell 1: 导入环境与全局配置 (Initialization)

In [ ]:
import sys
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.patches import Patch
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout,
                             QHBoxLayout, QFormLayout, QPushButton,
                             QLabel, QComboBox, QGroupBox, QScrollArea, QMessageBox,
                             QSlider, QCheckBox, QInputDialog, QGridLayout, QFileDialog)
from PySide6.QtCore import Qt
import xgboost
import lightgbm


def configure_matplotlib_chinese_font():
    candidates = [
        "Microsoft YaHei", "SimHei", "SimSun", "KaiTi", "FangSong",
        "Noto Sans CJK SC", "Source Han Sans SC", "Arial Unicode MS"
    ]
    installed_fonts = {font.name for font in fm.fontManager.ttflist}
    for font_name in candidates:
        if font_name in installed_fonts:
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [font_name, "DejaVu Sans"]
            plt.rcParams["axes.unicode_minus"] = False
            return font_name
    plt.rcParams["axes.unicode_minus"] = False
    return None


MATPLOTLIB_CJK_FONT = configure_matplotlib_chinese_font()

# ================== 全局常量 ==================
BASE_PATH = r"C:\Users\GIGABYTE\Desktop\Test\Analysis\新（可用）\CSV"
TARGET_KEYS = ["EUI", "sDA", "sGA", "UDI", "SVF"]
TARGET_DISPLAY = {
    "EUI": "EUI (kWh/m²)",
    "sDA": "sDA<sub>450/50%</sub> (%)",
    "sGA": "sGA<sub>0.35, 5%</sub> (%)",
    "UDI": "UDI<sub>100-2000lx</sub> (%)",
    "SVF": "SVF (%)"
}

FEATURE_ORDER = [
    'WWR_s_exp', 'WWR_s', 'H_s', 'W_s', 'SH_s', 'd_A_s', 'd_B_s',
    'WWR_c', 'H_c', 'W_c', 'SH_c', 'd_A_c', 'd_B_c',
    'F_BL', 'F_BR', 'F_TR', 'F_TL',
    'alpha_oh', 'L_oh', 'd_mv',
    'N_v', 'W_v', 'L_v', 'M_TL', 'M_TR'
]

FEATURE_META = {
    'WWR_s_exp': {'symbol': 'WWRs',  'label': '外窗侧窗墙比', 'unit': '%',  'min': 30,   'max': 70,   'step': 5},
    'WWR_s':     {'symbol': 'WWRs',  'label': '外窗侧窗墙比', 'unit': '%',  'min': 30,   'max': 70,   'step': 5},
    'H_s':       {'symbol': 'Hs',    'label': '窗高',         'unit': 'm',  'min': 1.2,  'max': 2.6,  'step': 0.1},
    'W_s':       {'symbol': 'Ws',    'label': '窗宽',         'unit': 'm',  'min': 4.0,  'max': 7.8,  'step': 0.1},
    'SH_s':      {'symbol': 'SHs',   'label': '窗台高度',     'unit': 'm',  'min': 0.4,  'max': 1.0,  'step': 0.1},
    'd_A_s':     {'symbol': 'dA,s',  'label': '采光窗左距',   'unit': 'm',  'min': 1.0,  'max': 8.0,  'step': 0.1},
    'd_B_s':     {'symbol': 'dB,s',  'label': '采光窗右距',   'unit': 'm',  'min': 0.2,  'max': 8.0,  'step': 0.1},
    'WWR_c':     {'symbol': 'WWRc',  'label': '走廊侧窗墙比', 'unit': '%',  'min': 10,   'max': 40,   'step': 5},
    'H_c':       {'symbol': 'Hc',    'label': '窗高',         'unit': 'm',  'min': 0.6,  'max': 2.0,  'step': 0.1},
    'W_c':       {'symbol': 'Wc',    'label': '窗宽',         'unit': 'm',  'min': 3.4,  'max': 6.0,  'step': 0.1},
    'SH_c':      {'symbol': 'SHc',   'label': '窗台高度',     'unit': 'm',  'min': 0.8,  'max': 2.0,  'step': 0.1},
    'd_A_c':     {'symbol': 'dA,c',  'label': '走廊窗左距',   'unit': 'm',  'min': 0.1,  'max': 6.0,  'step': 0.1},
    'd_B_c':     {'symbol': 'dB,c',  'label': '走廊窗右距',   'unit': 'm',  'min': 0.1,  'max': 6.0,  'step': 0.1},
    'F_BL':      {'symbol': 'FBL',   'label': '框底左宽度',   'unit': 'm',  'min': 0.1,  'max': 1.5,  'step': 0.1},
    'F_BR':      {'symbol': 'FBR',   'label': '框底右宽度',   'unit': 'm',  'min': 0.1,  'max': 1.5,  'step': 0.1},
    'F_TR':      {'symbol': 'FTR',   'label': '框顶右宽度',   'unit': 'm',  'min': 0.1,  'max': 1.5,  'step': 0.1},
    'F_TL':      {'symbol': 'FTL',   'label': '框顶左宽度',   'unit': 'm',  'min': 0.1,  'max': 1.5,  'step': 0.1},
    'alpha_oh':  {'symbol': 'αoh',   'label': '悬挑角度',     'unit': '°',  'min': -80,  'max': 80,   'step': 1},
    'L_oh':      {'symbol': 'Loh',   'label': '悬挑长度',     'unit': 'm',  'min': 0.1,  'max': 2.0,  'step': 0.1},
    'd_mv':      {'symbol': 'dmv',   'label': '悬挑移动距离', 'unit': 'm',  'min': 0.0,  'max': 3.8,  'step': 0.1},
    'N_v':       {'symbol': 'Nv',    'label': '垂直遮阳数量', 'unit': '片', 'min': 1,    'max': 10,   'step': 1},
    'W_v':       {'symbol': 'Wv',    'label': '垂直遮阳宽度', 'unit': 'm',  'min': 0.01, 'max': 1.2,  'step': 0.05,
                  'slider_values': [0.01] + [round(i * 0.05, 2) for i in range(1, 25)]},
    'L_v':       {'symbol': 'Lv',    'label': '垂直遮阳长度', 'unit': 'm',  'min': 0.1,  'max': 1.5,  'step': 0.1},
    'M_TL':      {'symbol': 'MTL',   'label': '左上移动距离', 'unit': 'm',  'min': 0.0,  'max': 1.2,  'step': 0.05},
    'M_TR':      {'symbol': 'MTR',   'label': '右上移动距离', 'unit': 'm',  'min': 0.0,  'max': 1.2,  'step': 0.05},
}

SHADE_FEATURE_MAP = {
    "基准 (Base)": FEATURE_ORDER[1:13],
    "框式 (Frame)": FEATURE_ORDER[1:13] + FEATURE_ORDER[13:17],
    "悬挑 (Overhang)": FEATURE_ORDER[1:13] + FEATURE_ORDER[17:20],
    "垂直 (Vertical)": FEATURE_ORDER[1:13] + FEATURE_ORDER[20:25],
    "组合 (O+V)": FEATURE_ORDER[1:13] + FEATURE_ORDER[17:25]
}

FEATURE_ALIAS = {
    'Frame_BL': 'F_BL', 'Frame_BR': 'F_BR',
    'Frame_TR': 'F_TR', 'Frame_TL': 'F_TL',
}

ORI_LIST = ["南向 (South 0°)", "北向 (North 0°)"]
SHADE_LIST = ["基准 (Base)", "悬挑 (Overhang)", "垂直 (Vertical)", "组合 (O+V)", "框式 (Frame)"]

PRESET_DATA = {
    (ORI_LIST[0], SHADE_LIST[0]): {'WWR_s':30,'H_s':2,'SH_s':0.9,'WWR_c':15,'H_c':0.9,'W_c':4.5,'SH_c':0.8},
    (ORI_LIST[0], SHADE_LIST[1]): {'WWR_s':30,'H_s':1.2,'SH_s':1,'WWR_c':15,'H_c':1.2,'W_c':3.4,'SH_c':0.8,'alpha_oh':20,'L_oh':1.9,'d_mv':0},
    (ORI_LIST[0], SHADE_LIST[2]): {'WWR_s_exp':31,'H_s':2.6,'SH_s':0.4,'WWR_c':10,'H_c':0.8,'W_c':3.4,'SH_c':2,'N_v':2,'W_v':1,'L_v':0.1,'M_TL':0,'M_TR':0.2},
    (ORI_LIST[0], SHADE_LIST[3]): {'WWR_s_exp':31,'H_s':1.4,'SH_s':0.8,'WWR_c':20,'H_c':0.9,'W_c':6,'SH_c':0.8,'alpha_oh':50,'L_oh':2,'d_mv':0,'N_v':7,'W_v':0.3,'L_v':0.2,'M_TL':0.15,'M_TR':0},
    (ORI_LIST[0], SHADE_LIST[4]): {'WWR_s':30,'H_s':2,'SH_s':1,'WWR_c':15,'H_c':1,'W_c':4,'SH_c':1,'F_BL':0.7,'F_BR':1.4,'F_TR':1.1,'F_TL':0.9},
    (ORI_LIST[1], SHADE_LIST[0]): {'WWR_s':40,'H_s':2.6,'SH_s':0.4,'WWR_c':10,'H_c':0.8,'W_c':3.4,'SH_c':2},
    (ORI_LIST[1], SHADE_LIST[1]): {'WWR_s':35,'H_s':2.2,'SH_s':0.8,'WWR_c':15,'H_c':0.9,'W_c':4.5,'SH_c':0.8,'alpha_oh':80,'L_oh':0.4,'d_mv':0.8},
    (ORI_LIST[1], SHADE_LIST[2]): {'WWR_s_exp':40,'H_s':1.9,'SH_s':1,'WWR_c':10,'H_c':0.8,'W_c':3.4,'SH_c':2,'N_v':8,'W_v':0.3,'L_v':0.1,'M_TL':0.1,'M_TR':0},
    (ORI_LIST[1], SHADE_LIST[3]): {'WWR_s_exp':40,'H_s':2.6,'SH_s':0.4,'WWR_c':10,'H_c':0.6,'W_c':4.5,'SH_c':2,'alpha_oh':-80,'L_oh':0.9,'d_mv':0.5,'N_v':5,'W_v':0.02,'L_v':0.1,'M_TL':0,'M_TR':0},
    (ORI_LIST[1], SHADE_LIST[4]): {'WWR_s':40,'H_s':2.6,'SH_s':0.4,'WWR_c':10,'H_c':0.8,'W_c':3.4,'SH_c':2,'F_BL':0.1,'F_BR':0.4,'F_TR':0.1,'F_TL':0.1}
}

VIEW_PRESETS = {
    '采光窗侧 (前视图)': {'elev': 0, 'azim': 0, 'roll': 0.0},
    '走廊窗侧 (后视图)': {'elev': 0, 'azim': 180, 'roll': 0.0},
    '透视图':            {'elev': 25, 'azim': 45, 'roll': 0.0},
}
VIEW_KEYS = list(VIEW_PRESETS.keys())

DOOR_ZONE = 1.4
CORRIDOR_USABLE = 9.0 - DOOR_ZONE * 2
MIN_D_A_S = 1.0
MIN_D_B_S = 0.2

# ================== 辅助函数 ==================

def _slider_int(real_val, step):
    return int(round(float(real_val) / float(step)))

def _slider_real(slider_int, step):
    return round(slider_int * float(step), 6)

def _format_val(real_val, step, unit):
    s = str(step)
    decimals = len(s.rstrip('0').split('.')[1]) if '.' in s else 0
    return f"{real_val:.{decimals}f} {unit}"

def _wv_slider_values():
    return FEATURE_META['W_v']['slider_values']

def _nearest_wv_index(real_val):
    values = _wv_slider_values()
    target = float(real_val)
    return min(range(len(values)), key=lambda idx: abs(values[idx] - target))

def _format_feature_value(feat, value):
    meta = FEATURE_META[feat]
    if feat == 'W_v':
        return f"{float(value):.2f} {meta['unit']}"
    step_text = str(meta['step'])
    decimals = len(step_text.rstrip('0').split('.')[1]) if '.' in step_text else 0
    return f"{float(value):.{decimals}f} {meta['unit']}"

def _scale_vertical_offsets_to_slider(mtl, mtr, wv_limit):
    step = float(FEATURE_META['M_TL']['step'])
    limit = round(max(0.0, float(wv_limit)), 2)
    total = max(0.0, float(mtl) + float(mtr))
    if total <= limit + 1e-9:
        return round(float(mtl), 2), round(float(mtr), 2)
    if limit < step:
        return 0.0, 0.0
    ratio = limit / max(total, 1e-9)
    scaled = [float(mtl) * ratio, float(mtr) * ratio]
    snapped = [round(round(val / step) * step, 2) for val in scaled]
    while snapped[0] + snapped[1] > limit + 1e-9:
        idx = 0 if snapped[0] >= snapped[1] else 1
        if snapped[idx] >= step:
            snapped[idx] = round(snapped[idx] - step, 2)
        elif snapped[1 - idx] >= step:
            snapped[1 - idx] = round(snapped[1 - idx] - step, 2)
        else:
            return 0.0, 0.0
    return max(0.0, snapped[0]), max(0.0, snapped[1])

def _resolve_offsets_with_mins(left_in, right_in, total_avail, min_left, min_right):
    total_avail = max(0.0, round(float(total_avail), 1))
    min_left = round(float(min_left), 1)
    min_right = round(float(min_right), 1)
    left_in = round(float(left_in), 1)
    right_in = round(float(right_in), 1)
    if total_avail <= 0:
        return min_left, min_right
    left_in = max(min_left, left_in)
    right_in = max(min_right, right_in)
    free = max(0.0, total_avail - min_left - min_right)
    if free <= 0:
        return round(min_left, 1), round(total_avail - min_left, 1)
    excess_left = left_in - min_left
    excess_right = right_in - min_right
    excess_total = excess_left + excess_right
    if excess_total <= 1e-9:
        fl = round(free / 2.0, 1)
        fr = round(free - fl, 1)
    else:
        fl = round(excess_left / excess_total * free, 1)
        fr = round(free - fl, 1)
    return round(min_left + fl, 1), round(min_right + fr, 1)

## Cell 2: 几何计算引擎 (Geometry Engine)

In [ ]:
def geometry_engine(ui, mode):
    wa = 27.0
    wl = 9.0
    wall_h = 3.8
    min_top_clearance = 0.8

    def safe_float(key, default=0.0):
        val = ui.get(key, default)
        try:
            return float(val) if val is not None else default
        except Exception:
            return default

    def skylight_height_cap(wwr_value):
        if abs(wwr_value - 30.0) < 1e-9:
            return 2.0
        if abs(wwr_value - 35.0) < 1e-9:
            return 2.2
        return 2.6

    def adjust_skylight_sill(hs_value, shs_value):
        shs_actual = min(max(round(shs_value, 2), 0.4), 1.0)
        top_gap = round(wall_h - hs_value - shs_actual, 2)
        while top_gap < min_top_clearance and shs_actual > 0.4:
            shs_actual = round(max(0.4, shs_actual - 0.1), 2)
            top_gap = round(wall_h - hs_value - shs_actual, 2)
        if top_gap < 0.05:
            top_gap = 0.0
        return round(shs_actual, 2), round(top_gap, 2)

    hs_in = safe_float('H_s', 2.0)
    hc = safe_float('H_c', 1.0)
    shs_in = safe_float('SH_s', 0.9)
    shc_in = safe_float('SH_c', 0.8)
    wv = safe_float('W_v', 0.1)

    L_orig = safe_float('L_oh', 1.0)
    d_orig = safe_float('d_mv', 0.5)
    angle = safe_float('alpha_oh', 0.0)

    wwr_c_in = safe_float('WWR_c', 15)
    max_area_c = 6.2 * hc
    max_wwr_c = (max_area_c / wa) * 100.0
    real_wwr_c = min(wwr_c_in, max_wwr_c)
    wc_actual = min((real_wwr_c / 100.0 * wa) / max(0.1, hc), 6.2)

    if hc <= 0.8:
        shc_actual = 2.0
    else:
        shc_actual = min(max(shc_in, 0.8), 1.0)

    is_vertical_mode = mode in ["垂直 (Vertical)", "组合 (O+V)"]
    design_wwr_s = safe_float('WWR_s_exp', 30) if is_vertical_mode else safe_float('WWR_s', 30)
    hs_cap = skylight_height_cap(design_wwr_s)
    hs_actual = min(max(round(hs_in, 1), 1.2), hs_cap)
    target_area_s = round((design_wwr_s / 100.0) * wa, 3)
    max_opening_w = round(max(0.0, wl - MIN_D_A_S - MIN_D_B_S), 3)

    total_opening_w = round(target_area_s / max(hs_actual, 0.1), 3)
    while total_opening_w > max_opening_w + 1e-9 and hs_actual < hs_cap:
        hs_actual = round(min(hs_actual + 0.1, hs_cap), 1)
        total_opening_w = round(target_area_s / max(hs_actual, 0.1), 3)
    total_opening_w = min(total_opening_w, max_opening_w)

    if is_vertical_mode:
        nv = max(2, int(safe_float('N_v', 2)))
        net_glass_w_limit = max(0.0, total_opening_w - (nv - 1) * wv)
        ws_pane = round(net_glass_w_limit / max(1, nv - 1), 1)
        while ws_pane > 0 and (ws_pane * (nv - 1) + (nv - 1) * wv) > max_opening_w + 1e-9:
            ws_pane = round(max(0.0, ws_pane - 0.1), 1)
        net_glass_w_actual = ws_pane * (nv - 1)
        total_opening_w_actual = net_glass_w_actual + (nv - 1) * wv
        actual_wwr_s = round((net_glass_w_actual * hs_actual / wa) * 100.0)
    else:
        nv = 0
        ws_pane = round(min(total_opening_w, max_opening_w), 1)
        while ws_pane > max_opening_w + 1e-9:
            ws_pane = round(max(0.0, ws_pane - 0.1), 1)
        total_opening_w_actual = ws_pane
        actual_wwr_s = round((ws_pane * hs_actual / wa) * 100.0)

    shs_actual, _ = adjust_skylight_sill(hs_actual, shs_in)

    max_d_mv = round(max(0.0, 0.8 + (wall_h - 0.8 - hs_actual - shs_actual)), 2)
    d_final = round(min(max(0.0, d_orig), max_d_mv), 2)

    L_final = L_orig
    if angle < 0:
        max_L_oh = round(d_final + shs_actual, 2)
        if L_final > max_L_oh:
            L_final = max_L_oh
            if L_final < 0:
                L_final = 0

    total_s = max(0.0, round(wl - total_opening_w_actual, 1))
    d_a_s_actual, d_b_s_actual = _resolve_offsets_with_mins(
        safe_float('d_A_s', MIN_D_A_S), safe_float('d_B_s', MIN_D_B_S),
        total_s, MIN_D_A_S, MIN_D_B_S)

    total_c = max(0.0, round(CORRIDOR_USABLE - wc_actual, 1))
    d_a_c_actual, d_b_c_actual = _resolve_offsets_with_mins(
        safe_float('d_A_c', total_c / 2.0), safe_float('d_B_c', total_c / 2.0),
        total_c, 0.1, 0.1)

    mtl = safe_float('M_TL', 0.0)
    mtr = safe_float('M_TR', 0.0)
    mtl, mtr = _scale_vertical_offsets_to_slider(mtl, mtr, wv)

    res = {k: safe_float(k) for k in FEATURE_ORDER}
    res.update({
        'H_s': hs_actual, 'WWR_s': int(actual_wwr_s), 'WWR_c': int(real_wwr_c),
        'W_s': ws_pane, 'W_c': round(wc_actual, 2),
        'SH_s': shs_actual, 'SH_c': round(shc_actual, 2),
        'd_A_s': d_a_s_actual, 'd_B_s': d_b_s_actual,
        'd_A_c': d_a_c_actual, 'd_B_c': d_b_c_actual,
        'N_v': nv, 'M_TL': mtl, 'M_TR': mtr,
        'L_oh': L_final, 'd_mv': d_final
    })
    return res

## Cell 3: 3D 绘图引擎与模型导出 (Visualization & Export)

In [ ]:
def draw_classroom_3d(ax, data, sha_mode, show_upper=False, show_side=False):
    ax.clear()
    L, D, H_total = 9.0, 9.0, 3.8
    ax.set_proj_type('ortho')
    ax.set_axis_off()

    def render_module(dy=0, dz=0, is_main=True):
        # 竖向线框：避免 plot3D(..., alpha<1)，mplot3d/Qt 后端下易缺线
        a_solid = 0.8 if is_main else 0.15
        a_glass = 0.5 if is_main else 0.15
        lw_main = 1.5 if is_main else 0.5
        c_edge = 'k' if is_main else 'gray'
        lw_vert = 1.15 if is_main else 0.85
        c_vert = '#1a1a1a' if is_main else '#9e9e9e'

        for z in [0, H_total]:
            ax.plot3D([0,9,9,0,0],[0+dy]*5,[z+dz]*5, c_edge, lw=lw_main)
        for x, y in [(0,0),(9,0),(9,9),(0,9)]:
            ax.plot3D([x,x],[y+dy,y+dy],[0+dz,H_total+dz], c_vert, lw=lw_vert)

        ax.add_collection3d(Poly3DCollection(
            [[(-3,0+dy,0+dz),(0,0+dy,0+dz),(0,9+dy,0+dz),(-3,9+dy,0+dz)]],
            facecolors='gray', alpha=0.1))
        ax.add_collection3d(Poly3DCollection(
            [[(-3,0+dy,3.0+dz),(0,0+dy,3.0+dz),(0,9+dy,3.0+dz),(-3,9+dy,3.0+dz)],
             [(-3,0+dy,3.8+dz),(0,0+dy,3.8+dz),(0,9+dy,3.8+dz),(-3,9+dy,3.8+dz)]],
            facecolors='gray', alpha=0.3))

        wc_d, hc_d, shc_d = data.get('W_c',4.0), data.get('H_c',1.0), data.get('SH_c',1.0)
        d_a_c = data.get('d_A_c', 0.0)
        yc = DOOR_ZONE + d_a_c
        ax.add_collection3d(Poly3DCollection(
            [[(0,yc+dy,shc_d+dz),(0,yc+wc_d+dy,shc_d+dz),
              (0,yc+wc_d+dy,shc_d+hc_d+dz),(0,yc+dy,shc_d+hc_d+dz)]],
            facecolors='lightgreen', edgecolors='g', alpha=a_glass, lw=1))

        for yd in [0.2, 9-0.2-1.2]:
            ax.add_collection3d(Poly3DCollection(
                [[(0,yd+dy,0+dz),(0,yd+1.2+dy,0+dz),(0,yd+1.2+dy,2.2+dz),(0,yd+dy,2.2+dz)]],
                facecolors='saddlebrown', edgecolors=c_edge, alpha=a_solid))

        bx1, bx2, bz1, bz2 = 2.0, 7.0, 0.9, 2.3
        ax.add_collection3d(Poly3DCollection(
            [[(bx1,0.02+dy,bz1+dz),(bx2,0.02+dy,bz1+dz),(bx2,0.02+dy,bz2+dz),(bx1,0.02+dy,bz2+dz)]],
            facecolors='#243b2f', edgecolors='black', alpha=0.9 if is_main else 0.2, lw=1))
        if is_main:
            ax.text((bx1+bx2)/2, 0.02+dy, bz2+0.15+dz, '黑板', color='black', ha='center', va='bottom', fontsize=10)

        is_v = "垂直" in sha_mode or "组合" in sha_mode
        nv = int(data.get('N_v', 0))
        wv_d, ws_d, hs_d, shs_d = data.get('W_v',0.1), data.get('W_s',4.0), data.get('H_s',2.0), data.get('SH_s',0.9)
        total_open = ((nv-1)*ws_d + (nv-1)*wv_d) if (is_v and nv >= 2) else ws_d
        ys = data.get('d_A_s', MIN_D_A_S)
        num_p = (nv-1) if (is_v and nv >= 2) else 1
        for i in range(num_p):
            cy = ys + wv_d/2 + i*(ws_d+wv_d) if (is_v and nv >= 2) else ys
            ax.add_collection3d(Poly3DCollection(
                [[(9,cy+dy,shs_d+dz),(9,cy+ws_d+dy,shs_d+dz),
                  (9,cy+ws_d+dy,shs_d+hs_d+dz),(9,cy+dy,shs_d+hs_d+dz)]],
                facecolors='skyblue', edgecolors='b', alpha=a_glass))

        if is_v and nv > 0:
            lv, mt_l, mt_r = data.get('L_v',0), data.get('M_TL',0), data.get('M_TR',0)
            for i in range(nv):
                by = ys + i*(ws_d+wv_d)
                hw = wv_d/2
                p1=(9,by-hw+dy,0+dz); p2=(9,by+hw+dy,0+dz)
                p3=(9,by+hw+dy,3.8+dz); p4=(9,by-hw+dy,3.8+dz)
                p5=(9+lv,by-hw+mt_l+dy,0+dz); p6=(9+lv,by+hw-mt_r+dy,0+dz)
                p7=(9+lv,by+hw-mt_r+dy,3.8+dz); p8=(9+lv,by-hw+mt_l+dy,3.8+dz)
                ax.add_collection3d(Poly3DCollection(
                    [[p1,p2,p3,p4],[p5,p6,p7,p8],[p1,p5,p8,p4],[p2,p6,p7,p3],[p1,p2,p6,p5],[p4,p3,p7,p8]],
                    facecolors='teal', alpha=a_solid, edgecolors='k', lw=0.85))

        if ("悬挑" in sha_mode or "组合" in sha_mode) and data.get('L_oh', 0) > 0:
            alpha_r = np.radians(data.get('alpha_oh', 0))
            loh, dmv = data['L_oh'], data.get('d_mv', 0)
            zt = 3.8 - dmv
            z_end = zt - loh * np.sin(alpha_r)
            x_ext = 9 + loh * np.cos(alpha_r)
            ax.add_collection3d(Poly3DCollection(
                [[(9,0+dy,zt+dz),(x_ext,0+dy,z_end+dz),(x_ext,9+dy,z_end+dz),(9,9+dy,zt+dz)]],
                facecolors='indianred', alpha=a_solid, edgecolors='darkred', lw=0.5))

        if "框式" in sha_mode:
            fbl,fbr,ftr,ftl = data.get('F_BL',0),data.get('F_BR',0),data.get('F_TR',0),data.get('F_TL',0)
            p_bl=(9,ys+dy,shs_d+dz); p_br=(9,ys+total_open+dy,shs_d+dz)
            p_tr=(9,ys+total_open+dy,shs_d+hs_d+dz); p_tl=(9,ys+dy,shs_d+hs_d+dz)
            e_bl=(9+fbl,ys+dy,shs_d+dz); e_br=(9+fbr,ys+total_open+dy,shs_d+dz)
            e_tr=(9+ftr,ys+total_open+dy,shs_d+hs_d+dz); e_tl=(9+ftl,ys+dy,shs_d+hs_d+dz)
            ax.add_collection3d(Poly3DCollection(
                [[p_bl,p_br,e_br,e_bl],[p_br,p_tr,e_tr,e_br],[p_tr,p_tl,e_tl,e_tr],[p_tl,p_bl,e_bl,e_tl]],
                facecolors='orange', alpha=a_glass, edgecolors='darkorange'))

        if is_main:
            ax.text(9, 4.5+dy, 4.1+dz, f"Actual WWR: {data['WWR_s']}%", color='darkred', fontweight='bold', ha='center')

    render_module(dy=0, dz=0, is_main=True)
    if show_upper: render_module(dy=0, dz=H_total, is_main=False)
    if show_side:  render_module(dy=D, dz=0, is_main=False)

    ax.set_xlim(-3.5, 11.5)
    ax.set_ylim(-2.5, 20.5 if show_side else 11.5)
    ax.set_zlim(0, 9 if show_upper else 5)
    x0,x1 = ax.get_xlim3d(); y0,y1 = ax.get_ylim3d(); z0,z1 = ax.get_zlim3d()
    ax.set_box_aspect((abs(x1-x0), abs(y1-y0), abs(z1-z0)))

    handles = [
        Patch(facecolor='gray', edgecolor='gray', alpha=0.3, label='走廊侧地坪/吊顶'),
        Patch(facecolor='lightgreen', edgecolor='g', alpha=0.5, label='走廊窗'),
        Patch(facecolor='skyblue', edgecolor='b', alpha=0.5, label='采光窗'),
        Patch(facecolor='saddlebrown', edgecolor='black', alpha=0.8, label='门'),
        Patch(facecolor='#243b2f', edgecolor='black', alpha=0.9, label='黑板')]
    if "垂直" in sha_mode or "组合" in sha_mode:
        handles.append(Patch(facecolor='teal', edgecolor='black', alpha=0.8, label='垂直遮阳'))
    if "悬挑" in sha_mode or "组合" in sha_mode:
        handles.append(Patch(facecolor='indianred', edgecolor='darkred', alpha=0.8, label='悬挑遮阳'))
    if "框式" in sha_mode:
        handles.append(Patch(facecolor='orange', edgecolor='darkorange', alpha=0.5, label='框式遮阳'))
    ax.figure.subplots_adjust(right=0.8)
    ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02,0.5),
              frameon=True, fontsize=9, title='颜色说明', title_fontsize=10)


# ================== OBJ 导出 ==================
OBJ_MATERIALS = {
    'wall_frame':       ((0.70, 0.70, 0.70), '主体框架'),
    'corridor_surface': ((0.55, 0.55, 0.55), '走廊侧地坪/吊顶'),
    'corridor_window':  ((0.56, 0.93, 0.56), '走廊窗'),
    'door':             ((0.55, 0.27, 0.07), '门'),
    'blackboard':       ((0.14, 0.23, 0.18), '黑板'),
    'skylight_window':  ((0.53, 0.81, 0.92), '采光窗'),
    'vertical_shading': ((0.00, 0.50, 0.50), '垂直遮阳'),
    'overhang_shading': ((0.80, 0.36, 0.36), '悬挑遮阳'),
    'frame_shading':    ((1.00, 0.65, 0.00), '框式遮阳'),
}

def export_classroom_obj(filepath, data, sha_mode):
    """导出教室三维模型为 OBJ + MTL，按图例分层。"""
    L, D, H = 9.0, 9.0, 3.8
    verts = []
    groups = {}

    def q(group, *pts):
        base = len(verts) + 1
        verts.extend(pts)
        groups.setdefault(group, []).append(tuple(range(base, base + len(pts))))

    # 主体 6 面
    q('wall_frame', (0,0,0),(L,0,0),(L,D,0),(0,D,0))
    q('wall_frame', (0,0,H),(0,D,H),(L,D,H),(L,0,H))
    q('wall_frame', (0,0,0),(L,0,0),(L,0,H),(0,0,H))
    q('wall_frame', (0,D,0),(L,D,0),(L,D,H),(0,D,H))
    q('wall_frame', (0,0,0),(0,D,0),(0,D,H),(0,0,H))
    q('wall_frame', (L,0,0),(L,D,0),(L,D,H),(L,0,H))

    # 走廊
    q('corridor_surface', (-3,0,0),(0,0,0),(0,D,0),(-3,D,0))
    q('corridor_surface', (-3,0,3.0),(0,0,3.0),(0,D,3.0),(-3,D,3.0))
    q('corridor_surface', (-3,0,3.8),(0,0,3.8),(0,D,3.8),(-3,D,3.8))

    # 走廊窗
    wc = data.get('W_c',4.0); hc = data.get('H_c',1.0)
    shc = data.get('SH_c',1.0); d_ac = data.get('d_A_c',0.0)
    yc = DOOR_ZONE + d_ac
    q('corridor_window', (0,yc,shc),(0,yc+wc,shc),(0,yc+wc,shc+hc),(0,yc,shc+hc))

    # 门
    for yd in [0.2, D-0.2-1.2]:
        q('door', (0,yd,0),(0,yd+1.2,0),(0,yd+1.2,2.2),(0,yd,2.2))

    # 黑板
    q('blackboard', (2.0,0.02,0.9),(7.0,0.02,0.9),(7.0,0.02,2.3),(2.0,0.02,2.3))

    # 采光窗
    is_v = "垂直" in sha_mode or "组合" in sha_mode
    nv = int(data.get('N_v',0))
    wv = data.get('W_v',0.1); ws = data.get('W_s',4.0)
    hs = data.get('H_s',2.0); shs = data.get('SH_s',0.9)
    total_open = ((nv-1)*ws + (nv-1)*wv) if (is_v and nv>=2) else ws
    ys = data.get('d_A_s', MIN_D_A_S)
    num_p = (nv-1) if (is_v and nv>=2) else 1
    for i in range(num_p):
        cy = ys + wv/2 + i*(ws+wv) if (is_v and nv>=2) else ys
        q('skylight_window', (L,cy,shs),(L,cy+ws,shs),(L,cy+ws,shs+hs),(L,cy,shs+hs))

    # 垂直遮阳
    if is_v and nv > 0:
        lv = data.get('L_v',0); mt_l = data.get('M_TL',0); mt_r = data.get('M_TR',0)
        for i in range(nv):
            by = ys + i*(ws+wv); hw = wv/2
            p1=(L,by-hw,0); p2=(L,by+hw,0); p3=(L,by+hw,H); p4=(L,by-hw,H)
            p5=(L+lv,by-hw+mt_l,0); p6=(L+lv,by+hw-mt_r,0)
            p7=(L+lv,by+hw-mt_r,H); p8=(L+lv,by-hw+mt_l,H)
            q('vertical_shading', p1,p2,p3,p4)
            q('vertical_shading', p5,p6,p7,p8)
            q('vertical_shading', p1,p5,p8,p4)
            q('vertical_shading', p2,p6,p7,p3)
            q('vertical_shading', p1,p2,p6,p5)
            q('vertical_shading', p4,p3,p7,p8)

    # 悬挑遮阳
    if ("悬挑" in sha_mode or "组合" in sha_mode) and data.get('L_oh',0) > 0:
        ar = np.radians(data.get('alpha_oh',0))
        loh = data['L_oh']; dmv = data.get('d_mv',0)
        zt = H - dmv; ze = zt - loh*np.sin(ar); xe = L + loh*np.cos(ar)
        q('overhang_shading', (L,0,zt),(xe,0,ze),(xe,D,ze),(L,D,zt))

    # 框式遮阳
    if "框式" in sha_mode:
        fbl,fbr,ftr,ftl = data.get('F_BL',0),data.get('F_BR',0),data.get('F_TR',0),data.get('F_TL',0)
        q('frame_shading',
          (L,ys,shs),(L,ys+total_open,shs),(L+fbr,ys+total_open,shs),(L+fbl,ys,shs))
        q('frame_shading',
          (L,ys+total_open,shs),(L,ys+total_open,shs+hs),(L+ftr,ys+total_open,shs+hs),(L+fbr,ys+total_open,shs))
        q('frame_shading',
          (L,ys,shs+hs),(L,ys+total_open,shs+hs),(L+ftr,ys+total_open,shs+hs),(L+ftl,ys,shs+hs))
        q('frame_shading',
          (L,ys,shs),(L,ys,shs+hs),(L+ftl,ys,shs+hs),(L+fbl,ys,shs))

    mtl_path = filepath.rsplit('.', 1)[0] + '.mtl'
    mtl_name = os.path.basename(mtl_path)

    with open(mtl_path, 'w', encoding='utf-8') as f:
        for name, ((r,g,b), label) in OBJ_MATERIALS.items():
            f.write(f"# {label}\nnewmtl {name}\n")
            f.write(f"Kd {r:.3f} {g:.3f} {b:.3f}\n")
            f.write(f"Ka {r*0.3:.3f} {g*0.3:.3f} {b*0.3:.3f}\n")
            f.write(f"Ks 0.1 0.1 0.1\nNs 10.0\nd 1.0\n\n")

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(f"# Classroom 3D Model\nmtllib {mtl_name}\n\n")
        for v in verts:
            f.write(f"v {v[0]:.4f} {v[2]:.4f} {v[1]:.4f}\n")
        f.write("\n")
        for gname, faces in groups.items():
            label = OBJ_MATERIALS.get(gname, (None, gname))[1]
            f.write(f"# {label}\ng {gname}\nusemtl {gname}\n")
            for face in faces:
                f.write("f " + " ".join(str(idx) for idx in face) + "\n")
            f.write("\n")

    return filepath, mtl_path


def _collect_mesh_data(data, sha_mode):
    """收集教室几何数据，供多种导出格式共用。"""
    L, D, H = 9.0, 9.0, 3.8
    meshes = {}

    def q(group, *pts):
        meshes.setdefault(group, {'verts': [], 'faces': []})
        m = meshes[group]
        base = len(m['verts'])
        m['verts'].extend(pts)
        if len(pts) == 4:
            m['faces'].append((base, base+1, base+2))
            m['faces'].append((base, base+2, base+3))
        elif len(pts) == 3:
            m['faces'].append((base, base+1, base+2))

    q('wall_frame', (0,0,0),(L,0,0),(L,D,0),(0,D,0))
    q('wall_frame', (0,0,H),(0,D,H),(L,D,H),(L,0,H))
    q('wall_frame', (0,0,0),(L,0,0),(L,0,H),(0,0,H))
    q('wall_frame', (0,D,0),(L,D,0),(L,D,H),(0,D,H))
    q('wall_frame', (0,0,0),(0,D,0),(0,D,H),(0,0,H))
    q('wall_frame', (L,0,0),(L,D,0),(L,D,H),(L,0,H))

    q('corridor_surface', (-3,0,0),(0,0,0),(0,D,0),(-3,D,0))
    q('corridor_surface', (-3,0,3.0),(0,0,3.0),(0,D,3.0),(-3,D,3.0))
    q('corridor_surface', (-3,0,3.8),(0,0,3.8),(0,D,3.8),(-3,D,3.8))

    wc = data.get('W_c',4.0); hc = data.get('H_c',1.0)
    shc = data.get('SH_c',1.0); d_ac = data.get('d_A_c',0.0)
    yc = DOOR_ZONE + d_ac
    q('corridor_window', (0,yc,shc),(0,yc+wc,shc),(0,yc+wc,shc+hc),(0,yc,shc+hc))

    for yd in [0.2, D-0.2-1.2]:
        q('door', (0,yd,0),(0,yd+1.2,0),(0,yd+1.2,2.2),(0,yd,2.2))

    q('blackboard', (2.0,0.02,0.9),(7.0,0.02,0.9),(7.0,0.02,2.3),(2.0,0.02,2.3))

    is_v = "垂直" in sha_mode or "组合" in sha_mode
    nv = int(data.get('N_v',0))
    wv = data.get('W_v',0.1); ws = data.get('W_s',4.0)
    hs = data.get('H_s',2.0); shs = data.get('SH_s',0.9)
    total_open = ((nv-1)*ws + (nv-1)*wv) if (is_v and nv>=2) else ws
    ys = data.get('d_A_s', MIN_D_A_S)
    num_p = (nv-1) if (is_v and nv>=2) else 1
    for i in range(num_p):
        cy = ys + wv/2 + i*(ws+wv) if (is_v and nv>=2) else ys
        q('skylight_window', (L,cy,shs),(L,cy+ws,shs),(L,cy+ws,shs+hs),(L,cy,shs+hs))

    if is_v and nv > 0:
        lv = data.get('L_v',0); mt_l = data.get('M_TL',0); mt_r = data.get('M_TR',0)
        for i in range(nv):
            by = ys + i*(ws+wv); hw = wv/2
            p1=(L,by-hw,0); p2=(L,by+hw,0); p3=(L,by+hw,H); p4=(L,by-hw,H)
            p5=(L+lv,by-hw+mt_l,0); p6=(L+lv,by+hw-mt_r,0)
            p7=(L+lv,by+hw-mt_r,H); p8=(L+lv,by-hw+mt_l,H)
            q('vertical_shading', p1,p2,p3,p4); q('vertical_shading', p5,p6,p7,p8)
            q('vertical_shading', p1,p5,p8,p4); q('vertical_shading', p2,p6,p7,p3)
            q('vertical_shading', p1,p2,p6,p5); q('vertical_shading', p4,p3,p7,p8)

    if ("悬挑" in sha_mode or "组合" in sha_mode) and data.get('L_oh',0) > 0:
        ar = np.radians(data.get('alpha_oh',0))
        loh = data['L_oh']; dmv = data.get('d_mv',0)
        zt = H - dmv; ze = zt - loh*np.sin(ar); xe = L + loh*np.cos(ar)
        q('overhang_shading', (L,0,zt),(xe,0,ze),(xe,D,ze),(L,D,zt))

    if "框式" in sha_mode:
        fbl,fbr,ftr,ftl = data.get('F_BL',0),data.get('F_BR',0),data.get('F_TR',0),data.get('F_TL',0)
        q('frame_shading', (L,ys,shs),(L,ys+total_open,shs),(L+fbr,ys+total_open,shs),(L+fbl,ys,shs))
        q('frame_shading', (L,ys+total_open,shs),(L,ys+total_open,shs+hs),(L+ftr,ys+total_open,shs+hs),(L+fbr,ys+total_open,shs))
        q('frame_shading', (L,ys,shs+hs),(L,ys+total_open,shs+hs),(L+ftr,ys+total_open,shs+hs),(L+ftl,ys,shs+hs))
        q('frame_shading', (L,ys,shs),(L,ys,shs+hs),(L+ftl,ys,shs+hs),(L+fbl,ys,shs))

    return meshes


def export_classroom_3ds(filepath, data, sha_mode):
    """导出教室三维模型为 .3ds (3D Studio) 格式，按图例分层。"""
    import struct
    meshes = _collect_mesh_data(data, sha_mode)

    def _build_mesh_chunk(name, verts, faces):
        buf = bytearray()
        def w16(v): buf.extend(struct.pack('<H', v))
        def w32(v): buf.extend(struct.pack('<I', v))
        def wf(v):  buf.extend(struct.pack('<f', v))

        obj_start = len(buf)
        w16(0x4000); w32(0)
        buf.extend(name[:10].encode('ascii','replace') + b'\x00')

        mesh_start = len(buf)
        w16(0x4100); w32(0)

        vlist_start = len(buf)
        w16(0x4110); w32(0)
        w16(len(verts))
        for x,y,z in verts:
            wf(x); wf(z); wf(y)
        struct.pack_into('<I', buf, vlist_start+2, len(buf)-vlist_start)

        flist_start = len(buf)
        w16(0x4120); w32(0)
        w16(len(faces))
        for a,b_,c in faces:
            w16(a); w16(b_); w16(c); w16(7)
        struct.pack_into('<I', buf, flist_start+2, len(buf)-flist_start)

        struct.pack_into('<I', buf, mesh_start+2, len(buf)-mesh_start)
        struct.pack_into('<I', buf, obj_start+2, len(buf)-obj_start)
        return bytes(buf)

    mesh_chunks = bytearray()
    for gname, m in meshes.items():
        if not m['verts'] or not m['faces']:
            continue
        mesh_chunks.extend(_build_mesh_chunk(gname, m['verts'], m['faces']))

    editor_chunk = bytearray()
    editor_chunk.extend(struct.pack('<H', 0x3D3D))
    editor_chunk.extend(struct.pack('<I', 6 + len(mesh_chunks)))
    editor_chunk.extend(mesh_chunks)

    ver = struct.pack('<HI', 0x0002, 10) + struct.pack('<I', 3)
    main_chunk = bytearray()
    main_chunk.extend(struct.pack('<H', 0x4D4D))
    main_chunk.extend(struct.pack('<I', 6 + len(ver) + len(editor_chunk)))
    main_chunk.extend(ver)
    main_chunk.extend(editor_chunk)

    with open(filepath, 'wb') as f:
        f.write(main_chunk)
    return filepath

## Cell 4: GUI 主程序 (Application)

In [ ]:
class SchemeComparisonWindow(QMainWindow):
    def __init__(self, parent_window=None):
        super().__init__(None)
        self.setWindowTitle("设计方案三维模型对比")
        if parent_window:
            self.resize(parent_window.size())
            self.move(parent_window.pos().x()+40, parent_window.pos().y()+40)
        else:
            self.resize(1500, 950)
        central = QWidget(); self.setCentralWidget(central)
        layout = QVBoxLayout(central)
        header = QLabel("左侧为基准方案  ←→  右侧为对比方案")
        header.setAlignment(Qt.AlignCenter)
        header.setStyleSheet("font-size: 16px; font-weight: bold; color: #2c3e50; padding: 6px;")
        layout.addWidget(header)
        canvas_row = QHBoxLayout()
        self.panels = {}
        for key, title_text in [('base','基准方案'),('current','对比方案')]:
            pl = QVBoxLayout()
            t = QLabel(title_text); t.setAlignment(Qt.AlignCenter)
            t.setStyleSheet("font-size: 14px; font-weight: bold; color: #34495e;")
            m = QLabel("--"); m.setAlignment(Qt.AlignCenter)
            m.setStyleSheet("color: #7f8c8d; font-size: 12px;")
            fig = plt.figure(figsize=(8,7)); ax = fig.add_subplot(111, projection='3d')
            canvas = FigureCanvas(fig)
            pl.addWidget(t); pl.addWidget(m); pl.addWidget(canvas, 1)
            canvas_row.addLayout(pl)
            self.panels[key] = {'meta':m,'figure':fig,'ax':ax,'canvas':canvas}
        layout.addLayout(canvas_row, 1)

    def set_schemes(self, base_scheme, current_scheme):
        for key, scheme in [('base',base_scheme),('current',current_scheme)]:
            p = self.panels[key]
            p['meta'].setText(f"{scheme['name']}  |  {scheme['ori']}  |  {scheme['shade']}")
            draw_classroom_3d(p['ax'], scheme['calculated'], scheme['shade'], False, False)
            p['ax'].view_init(elev=25, azim=45)
            p['canvas'].draw()


class ClassroomPredictorApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("青少年健康教室性能智能预测系统")
        self.resize(1500, 950)
        self.model_map = {}
        self.model_cache = {}
        self.inputs = {}
        self.rows = {}
        self.saved_schemes = []
        self.last_prediction = None
        self.compare_labels = {}
        self.compare_window = None
        self.view_axes = {}
        self.view_canvases = {}
        self._view_container = None
        self.scan_folders()
        self.init_ui()
        self.apply_presets()

    # ── 模型扫描 ──

    def scan_folders(self):
        if not os.path.exists(BASE_PATH):
            return
        folders = [d for d in os.listdir(BASE_PATH) if os.path.isdir(os.path.join(BASE_PATH, d))]
        for f in folders:
            ori = ORI_LIST[0] if "South0°" in f else (ORI_LIST[1] if "North0°" in f else None)
            if not ori:
                continue
            sha = SHADE_LIST[0]
            if "Overhang+Vertical" in f: sha = SHADE_LIST[3]
            elif "Overhang" in f: sha = SHADE_LIST[1]
            elif "Vertical" in f: sha = SHADE_LIST[2]
            elif "Frame" in f: sha = SHADE_LIST[4]
            self.model_map[(ori, sha)] = os.path.join(BASE_PATH, f)
        pref = os.path.join(BASE_PATH, "0312_2000_North0°_Overhang+Vertical(1)_replaced")
        if os.path.isdir(pref):
            self.model_map[(ORI_LIST[1], SHADE_LIST[3])] = pref

    # ── 滑杆读写 ──

    def get_feature_value(self, feat):
        if feat == 'W_v':
            idx = max(0, min(self.inputs[feat].value(), len(_wv_slider_values())-1))
            return float(_wv_slider_values()[idx])
        return _slider_real(self.inputs[feat].value(), FEATURE_META[feat]['step'])

    def set_feature_value(self, feat, real_val):
        if feat == 'W_v':
            slider = self.inputs[feat]
            idx = _nearest_wv_index(real_val)
            slider.blockSignals(True); slider.setValue(idx); slider.blockSignals(False)
            _, _, vl = self.rows[feat]
            if vl: vl.setText(_format_feature_value(feat, _wv_slider_values()[idx]))
            return
        slider = self.inputs[feat]; meta = FEATURE_META[feat]; step = meta['step']
        iv = max(slider.minimum(), min(slider.maximum(), _slider_int(real_val, step)))
        slider.blockSignals(True); slider.setValue(iv); slider.blockSignals(False)
        _, _, vl = self.rows[feat]
        if vl: vl.setText(_format_val(_slider_real(iv, step), step, meta['unit']))

    def invalidate_prediction(self, *args):
        self.last_prediction = None
        if hasattr(self, 'btn_save_scheme'):
            self.btn_save_scheme.setEnabled(False)

    # ── UI 构建 ──

    def init_ui(self):
        main_widget = QWidget(); self.setCentralWidget(main_widget)
        main_layout = QHBoxLayout(main_widget)

        scroll = QScrollArea(); scroll.setFixedWidth(460); scroll.setWidgetResizable(True)
        left = QWidget(); vbox = QVBoxLayout(left)

        grp_sel = QGroupBox("1. 策略选择"); f_sel = QFormLayout(grp_sel)
        self.cb_ori = QComboBox(); self.cb_sha = QComboBox()
        self.cb_ori.addItems(ORI_LIST); self.cb_sha.addItems(SHADE_LIST)
        self.cb_ori.currentTextChanged.connect(self.apply_presets)
        self.cb_sha.currentTextChanged.connect(self.apply_presets)
        self.cb_ori.currentTextChanged.connect(self.invalidate_prediction)
        self.cb_sha.currentTextChanged.connect(self.invalidate_prediction)
        self.cb_ori.currentTextChanged.connect(self._on_scheme_changed)
        self.cb_sha.currentTextChanged.connect(self._on_scheme_changed)
        f_sel.addRow("朝向:", self.cb_ori); f_sel.addRow("形式:", self.cb_sha)
        vbox.addWidget(grp_sel)

        grp_export = QGroupBox("导出三维模型 (OBJ / 3DS)")
        export_outer = QVBoxLayout(grp_export)
        export_row = QHBoxLayout()
        self.btn_export = QPushButton("导出 3D 模型")
        self.btn_export.setFixedHeight(42)
        self.btn_export.setToolTip("将当前几何导出为 OBJ+MTL 或 3DS；图层与图例一致。本区域在「策略选择」下方，无需滚到底部。")
        self.btn_export.setStyleSheet("background-color: #e67e22; color: white; font-weight: bold;")
        self.btn_export.clicked.connect(self.export_3d_model)
        self.cb_export_fmt = QComboBox()
        self.cb_export_fmt.addItems(["OBJ + MTL", "3DS (3D Studio)"])
        self.cb_export_fmt.setFixedHeight(42)
        self.cb_export_fmt.setFixedWidth(160)
        export_row.addWidget(self.btn_export)
        export_row.addWidget(self.cb_export_fmt)
        export_outer.addLayout(export_row)
        vbox.addWidget(grp_export)

        grp_param = QGroupBox("2. 设计变量"); self.param_form = QFormLayout(grp_param)
        for feat in FEATURE_ORDER:
            meta = FEATURE_META[feat]; step = meta['step']
            lbl = QLabel(f"{meta['symbol']}  {meta['label']}:")
            lbl.setToolTip(f"{meta['min']} ~ {meta['max']} {meta['unit']}  (步长 {step})")
            rw = QWidget(grp_param); rl = QHBoxLayout(rw); rl.setContentsMargins(0,0,0,0)
            slider = QSlider(Qt.Horizontal, rw)
            i_lo, i_hi = _slider_int(meta['min'], step), _slider_int(meta['max'], step)
            slider.setRange(i_lo, i_hi); slider.setSingleStep(1)
            slider.setTickPosition(QSlider.TicksBelow)
            slider.setTickInterval(max(1, (i_hi-i_lo)//10))
            vl = QLabel(_format_val(meta['min'], step, meta['unit']), rw)
            vl.setFixedWidth(80); vl.setStyleSheet("color: #2980b9; font-weight: bold;")
            def _mk(s=slider, l=vl, st=step, u=meta['unit']):
                def fn(v): l.setText(_format_val(_slider_real(v, st), st, u))
                return fn
            slider.valueChanged.connect(_mk())
            slider.valueChanged.connect(self.invalidate_prediction)
            rl.addWidget(slider, 1); rl.addWidget(vl)
            self.inputs[feat] = slider; self.rows[feat] = (lbl, rw, vl)
            self.param_form.addRow(lbl, rw)
        vbox.addWidget(grp_param)

        grp_disp = QGroupBox("3. 环境显示"); dl = QVBoxLayout()
        self.chk_upper = QCheckBox("显示纵向叠加层 (楼上)")
        self.chk_side  = QCheckBox("显示横向相邻间 (并排)")
        self.btn_reset_view = QPushButton("重置为默认视角")
        self.btn_reset_view.clicked.connect(self.reset_3d_view)
        self.chk_upper.stateChanged.connect(self.update_viz_only)
        self.chk_side.stateChanged.connect(self.update_viz_only)
        dl.addWidget(self.chk_upper); dl.addWidget(self.chk_side); dl.addWidget(self.btn_reset_view)

        sep = QLabel("── 多视图设置 ──"); sep.setAlignment(Qt.AlignCenter)
        sep.setStyleSheet("color: #7f8c8d; font-size: 11px; padding-top: 6px;"); dl.addWidget(sep)
        rc = QHBoxLayout(); rc.addWidget(QLabel("视图窗口数:"))
        self.cb_view_count = QComboBox()
        self.cb_view_count.addItems(["1 个视图","2 个视图","3 个视图"])
        self.cb_view_count.currentIndexChanged.connect(self._on_view_layout_changed)
        rc.addWidget(self.cb_view_count); dl.addLayout(rc)
        self.view_checks = {}
        for i, vk in enumerate(VIEW_KEYS):
            chk = QCheckBox(vk); chk.setChecked(i == 2)
            chk.stateChanged.connect(self._on_view_selection_changed)
            self.view_checks[vk] = chk; dl.addWidget(chk)
        grp_disp.setLayout(dl); vbox.addWidget(grp_disp)

        btn_row = QHBoxLayout()
        self.btn_run = QPushButton("⚡ 执行 Stacking 集成预测")
        self.btn_run.setFixedHeight(50)
        self.btn_run.setStyleSheet("background-color: #2c3e50; color: white; font-weight: bold;")
        self.btn_run.clicked.connect(self.do_prediction)
        self.btn_save_scheme = QPushButton("💾 保存当前方案")
        self.btn_save_scheme.setFixedHeight(50); self.btn_save_scheme.setEnabled(False)
        self.btn_save_scheme.setStyleSheet("background-color: #16a085; color: white; font-weight: bold;")
        self.btn_save_scheme.clicked.connect(self.save_current_scheme)
        btn_row.addWidget(self.btn_run); btn_row.addWidget(self.btn_save_scheme)
        vbox.addLayout(btn_row)

        vbox.addStretch()

        scroll.setWidget(left); main_layout.addWidget(scroll)
        right_box = QVBoxLayout()

        res_grp = QGroupBox("预测看板"); rg = QHBoxLayout()
        self.res_labels = {}
        for key in TARGET_KEYS:
            vu = QVBoxLayout()
            t = QLabel(TARGET_DISPLAY[key]); t.setAlignment(Qt.AlignCenter)
            lv = QLabel("--"); lv.setAlignment(Qt.AlignCenter)
            lv.setStyleSheet("font-size: 20px; font-weight: bold; color: #d35400;")
            self.res_labels[key] = lv; vu.addWidget(t); vu.addWidget(lv); rg.addLayout(vu)
        res_grp.setLayout(rg); right_box.addWidget(res_grp)

        cmp_grp = QGroupBox("方案保存与性能对比"); cl = QVBoxLayout()
        sr = QHBoxLayout()
        self.cb_compare_base = QComboBox(); self.cb_compare_current = QComboBox()
        self.cb_compare_base.currentTextChanged.connect(self.update_comparison_panel)
        self.cb_compare_current.currentTextChanged.connect(self.update_comparison_panel)
        sr.addWidget(QLabel("基准方案:")); sr.addWidget(self.cb_compare_base)
        sr.addWidget(QLabel("对比方案:")); sr.addWidget(self.cb_compare_current)
        cl.addLayout(sr)
        self.lbl_compare_hint = QLabel("请先执行预测，并至少保存两个方案后再进行对比。")
        self.lbl_compare_hint.setStyleSheet("color: #7f8c8d;"); cl.addWidget(self.lbl_compare_hint)
        cg = QGridLayout()
        cg.addWidget(QLabel("指标"),0,0); cg.addWidget(QLabel("基准方案"),0,1)
        cg.addWidget(QLabel("对比方案"),0,2); cg.addWidget(QLabel("变化值"),0,3)
        for ri, key in enumerate(TARGET_KEYS, 1):
            cg.addWidget(QLabel(TARGET_DISPLAY[key]), ri, 0)
            bl = QLabel("--"); cu = QLabel("--"); de = QLabel("--")
            for x in [bl,cu,de]: x.setAlignment(Qt.AlignCenter)
            de.setStyleSheet("font-weight: bold; color: #7f8c8d;")
            self.compare_labels[key] = {'base':bl,'current':cu,'delta':de}
            cg.addWidget(bl,ri,1); cg.addWidget(cu,ri,2); cg.addWidget(de,ri,3)
        cl.addLayout(cg)
        self.btn_open_compare_window = QPushButton("🧭 打开三维方案对比窗口")
        self.btn_open_compare_window.setFixedHeight(42); self.btn_open_compare_window.setEnabled(False)
        self.btn_open_compare_window.setStyleSheet("background-color: #8e44ad; color: white; font-weight: bold;")
        self.btn_open_compare_window.clicked.connect(self.open_comparison_window)
        cl.addWidget(self.btn_open_compare_window)
        cmp_grp.setLayout(cl); right_box.addWidget(cmp_grp)

        self.fig = plt.figure(figsize=(9,9))
        self.ax = self.fig.add_subplot(111, projection='3d')
        self.ax.view_init(elev=25, azim=45)
        self.canvas = FigureCanvas(self.fig)
        right_box.addWidget(self.canvas)
        main_layout.addLayout(right_box)

        # W_v 特殊刻度同步
        if 'W_v' in self.inputs:
            sl = self.inputs['W_v']; _, _, vlbl = self.rows['W_v']
            def _sync_wv(v):
                if vlbl is None: return
                idx = max(0, min(int(v), len(_wv_slider_values())-1))
                vlbl.setText(_format_feature_value('W_v', _wv_slider_values()[idx]))
            sl.valueChanged.connect(_sync_wv); _sync_wv(sl.value())

        for w in self.inputs.values():
            if hasattr(w, 'valueChanged'):
                w.valueChanged.connect(lambda _v, s=self: s.update_viz_only(preserve_limits=True))

        self._orig_canvas = self.canvas; self._orig_fig = self.fig
        self._right_layout = None
        for i in range(main_layout.count()):
            item = main_layout.itemAt(i)
            if item and item.layout():
                lay = item.layout()
                for j in range(lay.count()):
                    sub = lay.itemAt(j)
                    if sub and sub.widget() is self._orig_canvas:
                        self._right_layout = lay; break
                if self._right_layout: break
        self._rebuild_view_canvas()

    # ── 3D 视角 ──

    def capture_3d_view_state(self):
        return {'elev':self.ax.elev,'azim':self.ax.azim,
                'roll':getattr(self.ax,'roll',0.0),
                'xlim':self.ax.get_xlim3d(),'ylim':self.ax.get_ylim3d(),'zlim':self.ax.get_zlim3d()}

    def restore_3d_view_state(self, state, restore_limits=False):
        if not state: return
        try: self.ax.view_init(elev=state['elev'], azim=state['azim'], roll=state.get('roll',0.0))
        except TypeError: self.ax.view_init(elev=state['elev'], azim=state['azim'])
        if restore_limits:
            self.ax.set_xlim3d(*state['xlim']); self.ax.set_ylim3d(*state['ylim']); self.ax.set_zlim3d(*state['zlim'])

    def _on_scheme_changed(self, *a):
        self.reset_3d_view()

    def reset_3d_view(self, *a):
        self._redraw_all_views()

    def update_viz_only(self, *a, preserve_limits=False):
        sha = self.cb_sha.currentText()
        raw = {f: self.get_feature_value(f) for f in FEATURE_ORDER}
        cal = geometry_engine(raw, sha)
        su, ss = self.chk_upper.isChecked(), self.chk_side.isChecked()
        pk = VIEW_KEYS[2]; sv = self.capture_3d_view_state() if pk in self.view_axes else None
        for vk, ax in self.view_axes.items():
            draw_classroom_3d(ax, cal, sha, su, ss)
            if vk == pk and sv: self.restore_3d_view_state(sv, restore_limits=preserve_limits)
            else:
                pr = VIEW_PRESETS[vk]
                try: ax.view_init(elev=pr['elev'], azim=pr['azim'], roll=pr.get('roll',0.0))
                except TypeError: ax.view_init(elev=pr['elev'], azim=pr['azim'])
        for _, (_, c) in self.view_canvases.items(): c.draw_idle()

    # ── 多视图管理 ──

    def _on_view_layout_changed(self, idx):
        count = idx + 1
        checked = [k for k in VIEW_KEYS if self.view_checks[k].isChecked()]
        if len(checked) > count:
            for k in reversed(checked[count:]):
                self.view_checks[k].blockSignals(True); self.view_checks[k].setChecked(False); self.view_checks[k].blockSignals(False)
        elif len(checked) < count:
            for k in VIEW_KEYS:
                if len(checked) >= count: break
                if k not in checked:
                    self.view_checks[k].blockSignals(True); self.view_checks[k].setChecked(True); self.view_checks[k].blockSignals(False)
                    checked.append(k)
        self._rebuild_view_canvas()

    def _on_view_selection_changed(self, state):
        count = self.cb_view_count.currentIndex() + 1
        checked = [k for k in VIEW_KEYS if self.view_checks[k].isChecked()]
        if len(checked) > count:
            sender = self.sender()
            for k in VIEW_KEYS:
                if self.view_checks[k] is not sender and self.view_checks[k].isChecked() and len(checked) > count:
                    self.view_checks[k].blockSignals(True); self.view_checks[k].setChecked(False); self.view_checks[k].blockSignals(False)
                    checked.remove(k)
        if not checked:
            self.view_checks[VIEW_KEYS[0]].blockSignals(True); self.view_checks[VIEW_KEYS[0]].setChecked(True); self.view_checks[VIEW_KEYS[0]].blockSignals(False)
        self._rebuild_view_canvas()

    def _active_view_keys(self):
        return [k for k in VIEW_KEYS if self.view_checks[k].isChecked()] or [VIEW_KEYS[2]]

    def _rebuild_view_canvas(self):
        active = self._active_view_keys()
        old_figs = [fig for _, (fig, _) in self.view_canvases.items()]
        if self._view_container is not None:
            if self._right_layout: self._right_layout.removeWidget(self._view_container)
            self._view_container.setParent(None); self._view_container.deleteLater()
            self._view_container = None
        if self._orig_canvas is not None and self._orig_canvas.parent() is not None:
            if self._right_layout: self._right_layout.removeWidget(self._orig_canvas)
            self._orig_canvas.hide(); self._orig_canvas.setParent(None)
        self._view_container = QWidget()
        cl = QVBoxLayout(self._view_container) if len(active)==1 else QHBoxLayout(self._view_container)
        cl.setContentsMargins(0,0,0,0); cl.setSpacing(4)
        self.view_axes = {}; self.view_canvases = {}
        fw = max(4, 9//len(active)); fh = 7 if len(active)<=2 else 6
        for vk in active:
            pl = QVBoxLayout()
            t = QLabel(vk); t.setAlignment(Qt.AlignCenter)
            t.setStyleSheet("font-size: 11px; font-weight: bold; color: #34495e; padding: 2px;")
            pl.addWidget(t)
            fig = plt.figure(figsize=(fw,fh)); ax = fig.add_subplot(111, projection='3d')
            canvas = FigureCanvas(fig); pl.addWidget(canvas, 1); cl.addLayout(pl)
            self.view_axes[vk] = ax; self.view_canvases[vk] = (fig, canvas)
        pk = VIEW_KEYS[2]
        if pk in self.view_axes:
            self.ax = self.view_axes[pk]; self.fig, self.canvas = self.view_canvases[pk]
        else:
            fk = active[0]; self.ax = self.view_axes[fk]; self.fig, self.canvas = self.view_canvases[fk]
        if self._right_layout: self._right_layout.addWidget(self._view_container)
        else: self.centralWidget().layout().addWidget(self._view_container)
        for f in old_figs:
            try: plt.close(f)
            except: pass
        self._redraw_all_views()

    def _redraw_all_views(self):
        sha = self.cb_sha.currentText()
        raw = {f: self.get_feature_value(f) for f in FEATURE_ORDER}
        cal = geometry_engine(raw, sha)
        su, ss = self.chk_upper.isChecked(), self.chk_side.isChecked()
        for vk, ax in self.view_axes.items():
            draw_classroom_3d(ax, cal, sha, su, ss)
            pr = VIEW_PRESETS[vk]
            try: ax.view_init(elev=pr['elev'], azim=pr['azim'], roll=pr.get('roll',0.0))
            except TypeError: ax.view_init(elev=pr['elev'], azim=pr['azim'])
        for _, (_, c) in self.view_canvases.items(): c.draw_idle()

    # ── 预设 / 可见性 ──

    def apply_presets(self):
        ori, sha = self.cb_ori.currentText(), self.cb_sha.currentText()
        preset = PRESET_DATA.get((ori, sha), {})
        for feat, val in preset.items():
            if feat in self.inputs:
                self.set_feature_value(feat, float(val))
        active_list = SHADE_FEATURE_MAP.get(sha, FEATURE_ORDER)
        is_p = "垂直" in sha or "组合" in sha
        for f, (lbl, rw, vl) in self.rows.items():
            vis = f in active_list or (f == 'WWR_s_exp' and is_p)
            lbl.setVisible(vis); rw.setVisible(vis)
            if f == 'WWR_s': self.inputs[f].setEnabled(not is_p)

    # ── 方案保存与对比 ──

    def format_metric(self, key, value):
        return f"{float(value):.2f}"

    def clear_comparison_panel(self, hint):
        self.lbl_compare_hint.setText(hint)
        for ls in self.compare_labels.values():
            ls['base'].setText("--"); ls['current'].setText("--"); ls['delta'].setText("--")
            ls['delta'].setStyleSheet("font-weight: bold; color: #7f8c8d;")

    def get_saved_scheme(self, name):
        return next((s for s in self.saved_schemes if s['name'] == name), None)

    def make_scheme_name_unique(self, base_name):
        names = {s['name'] for s in self.saved_schemes}
        if base_name not in names: return base_name
        idx = 2
        while f"{base_name} ({idx})" in names: idx += 1
        return f"{base_name} ({idx})"

    def refresh_saved_scheme_selectors(self, base_name=None, current_name=None):
        names = [s['name'] for s in self.saved_schemes]
        for cb in [self.cb_compare_base, self.cb_compare_current]:
            cb.blockSignals(True); cb.clear(); cb.addItems(names); cb.blockSignals(False)
        if names:
            if len(names) >= 2:
                self.cb_compare_base.setCurrentText(base_name if base_name in names else names[-2])
                self.cb_compare_current.setCurrentText(current_name if current_name in names else names[-1])
            else:
                self.cb_compare_base.setCurrentText(names[0])
                self.cb_compare_current.setCurrentText(names[0])
        self.update_comparison_panel()
        if hasattr(self, 'btn_open_compare_window'):
            self.btn_open_compare_window.setEnabled(len(self.saved_schemes) >= 2)

    def update_comparison_panel(self, *a):
        if len(self.saved_schemes) < 2:
            self.clear_comparison_panel("请至少保存两个方案后再进行性能对比。"); return
        bn = self.cb_compare_base.currentText().strip()
        cn = self.cb_compare_current.currentText().strip()
        if not bn or not cn: self.clear_comparison_panel("请选择两个已保存方案。"); return
        if bn == cn: self.clear_comparison_panel("请选择两个不同的已保存方案进行对比。"); return
        bs = self.get_saved_scheme(bn); cs = self.get_saved_scheme(cn)
        if not bs or not cs: self.clear_comparison_panel("方案数据不存在。"); return
        self.lbl_compare_hint.setText(f"变化值 = 对比方案 - 基准方案 | {bn} vs {cn}")
        for key in TARGET_KEYS:
            bv = float(bs['results'][key]); cv = float(cs['results'][key]); dv = cv - bv
            dc = "#7f8c8d" if abs(dv) < 1e-9 else ("#d35400" if dv > 0 else "#2980b9")
            self.compare_labels[key]['base'].setText(self.format_metric(key, bv))
            self.compare_labels[key]['current'].setText(self.format_metric(key, cv))
            self.compare_labels[key]['delta'].setText(f"{dv:+.2f}")
            self.compare_labels[key]['delta'].setStyleSheet(f"font-weight: bold; color: {dc};")
        if hasattr(self, 'btn_open_compare_window'):
            self.btn_open_compare_window.setEnabled(len(self.saved_schemes) >= 2)
        if self.compare_window and self.compare_window.isVisible():
            self._sync_comparison_window()

    def save_current_scheme(self):
        if not self.last_prediction:
            QMessageBox.information(self, "暂无可保存方案", "请先执行一次预测。"); return
        dn = f"方案{len(self.saved_schemes)+1} - {self.last_prediction['ori'].split()[0]} / {self.last_prediction['shade'].split()[0]}"
        name, ok = QInputDialog.getText(self, "保存设计方案", "请输入方案名称：", text=dn)
        if not ok: return
        name = self.make_scheme_name_unique((name or dn).strip() or dn)
        prev = self.saved_schemes[-1]['name'] if self.saved_schemes else None
        self.saved_schemes.append({
            'name': name, 'ori': self.last_prediction['ori'], 'shade': self.last_prediction['shade'],
            'raw_inputs': dict(self.last_prediction['raw_inputs']),
            'calculated': dict(self.last_prediction['calculated']),
            'results': dict(self.last_prediction['results'])})
        self.refresh_saved_scheme_selectors(base_name=prev or name, current_name=name)
        msg = f"{name} 已保存。" + ("继续保存新方案后可对比。" if len(self.saved_schemes)==1 else "已与上一方案建立对比。")
        QMessageBox.information(self, "方案已保存", msg)

    def open_comparison_window(self):
        if len(self.saved_schemes) < 2:
            QMessageBox.information(self, "无法对比", "请至少保存两个方案。"); return
        bn = self.cb_compare_base.currentText().strip()
        cn = self.cb_compare_current.currentText().strip()
        if not bn or not cn or bn == cn:
            QMessageBox.information(self, "无法对比", "请选择两个不同的方案。"); return
        if self.compare_window is None:
            self.compare_window = SchemeComparisonWindow(self)
        else: self.compare_window.resize(self.size())
        self._sync_comparison_window()
        self.compare_window.show(); self.compare_window.raise_(); self.compare_window.activateWindow()

    def _sync_comparison_window(self):
        if not self.compare_window: return
        bs = self.get_saved_scheme(self.cb_compare_base.currentText().strip())
        cs = self.get_saved_scheme(self.cb_compare_current.currentText().strip())
        if bs and cs and bs['name'] != cs['name']:
            self.compare_window.set_schemes(bs, cs)

    # ── 3D 模型导出 ──

    def export_3d_model(self):
        sha = self.cb_sha.currentText()
        raw = {f: self.get_feature_value(f) for f in FEATURE_ORDER}
        cal = geometry_engine(raw, sha)
        fmt = self.cb_export_fmt.currentText()
        if "3DS" in fmt:
            path, _ = QFileDialog.getSaveFileName(
                self, "导出 3D 模型", "classroom_model.3ds", "3DS Files (*.3ds)")
            if not path: return
            try:
                out = export_classroom_3ds(path, cal, sha)
                QMessageBox.information(self, "导出成功",
                    f"3DS 模型已导出（按图例分层）：\n\n"
                    f"模型文件：{out}\n\n"
                    f"可直接导入 3ds Max / Blender / Rhino 等软件。\n"
                    f"各图层在导入后将自动显示为独立对象。")
            except Exception as e:
                QMessageBox.critical(self, "导出失败", f"导出过程出错：\n{str(e)}")
        else:
            path, _ = QFileDialog.getSaveFileName(
                self, "导出 3D 模型", "classroom_model.obj", "OBJ Files (*.obj)")
            if not path: return
            try:
                obj_path, mtl_path = export_classroom_obj(path, cal, sha)
                QMessageBox.information(self, "导出成功",
                    f"OBJ 模型已导出（按图例分层）：\n\n"
                    f"模型文件：{obj_path}\n材质文件：{mtl_path}\n\n"
                    f"可直接导入 3ds Max / Blender / Rhino / SketchUp 等软件。")
            except Exception as e:
                QMessageBox.critical(self, "导出失败", f"导出过程出错：\n{str(e)}")

    # ── 核心预测 ──

    def do_prediction(self):
        ori_mode = self.cb_ori.currentText()
        sha_mode = self.cb_sha.currentText()
        folder = self.model_map.get((ori_mode, sha_mode))
        if not folder:
            QMessageBox.warning(self, "模型缺失", "当前朝向与遮阳形式尚未找到对应模型目录。"); return
        self.last_prediction = None; self.btn_save_scheme.setEnabled(False)
        try:
            self.btn_run.setEnabled(False); QApplication.processEvents()
            raw_in = {f: self.get_feature_value(f) for f in FEATURE_ORDER}
            cal_data = geometry_engine(raw_in, sha_mode)

            if "悬挑" in sha_mode or "组合" in sha_mode:
                id_dmv = self.get_feature_value('d_mv')
                if id_dmv > cal_data['d_mv'] + 1e-9:
                    QMessageBox.warning(self, "悬挑移动约束",
                        f"dmv 不能超过 0.8 + (3.8 - 0.8 - Hs - SHs)\n\n"
                        f"  计算后 Hs={cal_data['H_s']}, SHs={cal_data['SH_s']}\n"
                        f"  最大允许值={round(max(0,0.8+(3.8-0.8-cal_data['H_s']-cal_data['SH_s'])),2)}\n"
                        f"  您设定的 dmv={id_dmv}\n\ndmv 已自动限制为 {cal_data['d_mv']}")
                a_val = self.get_feature_value('alpha_oh')
                if a_val < 0:
                    ml = round(cal_data['d_mv']+cal_data['SH_s'], 2)
                    il = self.get_feature_value('L_oh')
                    if il > ml:
                        QMessageBox.warning(self, "遮阳板长度约束",
                            f"当 αoh<0 时，Loh 必须 ≤ dmv+SHs\n\n  最大允许值={ml}\n  您设定的 Loh={il}\n\nLoh 已自动限制为 {cal_data['L_oh']}")

            if "垂直" in sha_mode or "组合" in sha_mode:
                im = raw_in.get('M_TL',0); ir = raw_in.get('M_TR',0); wvv = raw_in.get('W_v',0.01)
                if im+ir > wvv:
                    QMessageBox.warning(self, "垂直遮阳移动约束",
                        f"MTL+MTR 之和不能超过 Wv\n\n  Wv={_format_feature_value('W_v',wvv)}\n"
                        f"  MTL={_format_feature_value('M_TL',im)}, MTR={_format_feature_value('M_TR',ir)}\n"
                        f"  合计={im+ir:.2f}m > Wv\n\n已自动缩放：MTL={_format_feature_value('M_TL',cal_data['M_TL'])}, MTR={_format_feature_value('M_TR',cal_data['M_TR'])}")

            for key in ['WWR_c','WWR_s','H_s','W_s','W_c','SH_s','SH_c','d_A_s','d_B_s','d_A_c','d_B_c','L_oh','d_mv','M_TL','M_TR']:
                if key in cal_data: self.set_feature_value(key, cal_data[key])

            ckey = (ori_mode, sha_mode)
            if ckey not in self.model_cache:
                cache = {}
                for t in TARGET_KEYS:
                    for m in ["xgb","lgbm","rf","meta"]:
                        cache[f"{t}_{m}"] = joblib.load(os.path.join(folder, f"{m}_model_{t}.joblib"))
                xtp = os.path.join(folder, "X_train.joblib")
                cache['_feature_order'] = joblib.load(xtp).columns.tolist() if os.path.exists(xtp) else None
                self.model_cache[ckey] = cache

            mc = self.model_cache[ckey]
            model_feats = mc.get('_feature_order')
            if model_feats:
                row = [cal_data.get(FEATURE_ALIAS.get(f, f), 0.0) for f in model_feats]
                X = pd.DataFrame([row], columns=model_feats)
            else:
                af = SHADE_FEATURE_MAP.get(sha_mode, FEATURE_ORDER[1:13])
                X = pd.DataFrame([[cal_data[f] for f in af]], columns=af)

            result_values = {}
            for t in TARGET_KEYS:
                m1 = mc[f"{t}_xgb"].predict(X)
                m2 = mc[f"{t}_lgbm"].predict(X)
                m3 = mc[f"{t}_rf"].predict(X)
                final = mc[f"{t}_meta"].predict(np.column_stack([m1,m2,m3]))[0]
                if t in ["sDA","sGA"]:
                    final = max(0.0, min(final, 1.0)); v_show = final * 100
                elif t in ["UDI","SVF"]:
                    v_show = max(0.0, min(final, 100.0))
                else: v_show = final
                result_values[t] = float(v_show)
                self.res_labels[t].setText(f"{v_show:.2f}")

            pk = VIEW_KEYS[2]; sv = self.capture_3d_view_state() if pk in self.view_axes else None
            su, ss = self.chk_upper.isChecked(), self.chk_side.isChecked()
            for vk, ax in self.view_axes.items():
                draw_classroom_3d(ax, cal_data, sha_mode, su, ss)
                if vk == pk and sv: self.restore_3d_view_state(sv, restore_limits=True)
                else:
                    pr = VIEW_PRESETS[vk]
                    try: ax.view_init(elev=pr['elev'], azim=pr['azim'], roll=pr.get('roll',0.0))
                    except TypeError: ax.view_init(elev=pr['elev'], azim=pr['azim'])
            for _, (_, c) in self.view_canvases.items(): c.draw_idle()

            self.last_prediction = {'ori':ori_mode,'shade':sha_mode,'raw_inputs':dict(raw_in),'calculated':dict(cal_data),'results':result_values}
            self.btn_save_scheme.setEnabled(True); self.btn_run.setEnabled(True)
        except Exception as e:
            import traceback; traceback.print_exc()
            QMessageBox.critical(self, "运行故障", f"预测或计算出错：\n{str(e)}")
            self.btn_save_scheme.setEnabled(False); self.btn_run.setEnabled(True)

## Cell 5: 启动执行 (Execution)

In [ ]:
if __name__ == "__main__":
    app = QApplication.instance() or QApplication(sys.argv)
    main_win = ClassroomPredictorApp()
    main_win.show()
    app.exec()